# 02 - Mạng CNN Pretrained và Tinh chỉnh (Fine-tuning)

Trong bài học này, chúng ta tiếp cận mô hình học sâu:
- Tìm hiểu cấu trúc Backbone tiền huấn luyện trên ImageNet (DenseNet-121) và Classification Head xuất 1 logit cho mỗi ảnh.
- Quy ước nhãn cặp $[1-y, y]$, logits $s_0, s_1$ và hàm mất mát Binary Cross-Entropy.
- Khái niệm thực nghiệm: **BatchNorm eval mode** và **Head-only Warmup**.
- Trực quan hóa bước huấn luyện `train_step` trên một batch ảnh thật bằng chính hàm chia sẻ của core trainer.
- Huấn luyện và so sánh trực tiếp trên Fold 0: `Frozen` vs `center60` (full fine-tuning).
- Phân tích lỗi chi tiết trên tập validation.

In [ ]:
from pathlib import Path
import os
import sys
import inspect

def find_task_root():
    cwd = Path.cwd().resolve()
    for cand in [cwd, cwd.parent, cwd / 'KeMaoDanh', cwd.parent / 'KeMaoDanh']:
        if (cand / 'src/kmd').is_dir() and (cand / 'configs').is_dir():
            return cand.resolve()
    raise FileNotFoundError("Mở notebook từ repo root, KeMaoDanh hoặc KeMaoDanh/notebooks.")

TASK_ROOT = find_task_root()
if str(TASK_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(TASK_ROOT / 'src'))

import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from kmd.core import PACKAGE, read_csv, split_fold, metric, read_json
from kmd.config import Config
from kmd.data import Pairs, make_loader
from kmd.models import model_for, train_mode, head_name, logits, objective, predict_pairs
from kmd.training import train_step
from kmd.pipeline import prepare_development, load_session, train_cnn_fold, aligned_predictions

RUN_ID = 'lesson_session'
print(f"Phiên làm việc: {RUN_ID} | PyTorch version: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

## 1. Kiến trúc Backbone DenseNet-121 và Classification Head

Mô hình gồm 2 phần:
- **Backbone (DenseNet-121):** Trích xuất vector đặc trưng $1024$-chiều từ ảnh.
- **Classification Head:** Một lớp tuyến tính duy nhất `nn.Linear(1024, 1)` xuất ra **1 giá trị logit ảnh $s$**.

Với cặp ảnh gồm `image_0` và `image_1`, mô hình tính ra $s_0$ và $s_1$. Xác suất dự đoán ảnh 1 là ảnh giả là:
$$p = \sigma(s_1 - s_0) = \frac{1}{1 + e^{-(s_1 - s_0)}}$$

Pretrained nghĩa là backbone đã học tham số từ ImageNet trước khi thấy bài toán này; head mới được khởi tạo và cần học từ nhãn của chúng ta. Logit là điểm số thực chưa bị giới hạn vào [0,1]. Nếu s₁>s₀ thì p>0.5, mô hình chọn ảnh 1.

Vì y là vị trí ảnh giả, nhãn riêng của hai ảnh là [1-y, y]: y=0 cho [1,0], y=1 cho [0,1]. Cấu hình này dùng binary cross-entropy trên **hai logit ảnh**, khuyến khích logit ảnh giả cao và ảnh thật thấp. Xác suất cặp dùng hiệu hai logit khi dự đoán; đây không phải loss cặp. Cell dưới in chính hàm objective để thấy khác biệt.

## 2. Quy ước thực nghiệm: BatchNorm eval mode và Head Warmup

- **BatchNorm eval mode:** Khi fine-tune trên tập dữ liệu nhỏ với microbatch nhỏ, việc cập nhật thống kê trung bình/phương sai trượt (`running_mean`, `running_var`) có thể gây dao động lớn. Quy ước trong thực nghiệm này là đặt toàn bộ lớp BatchNorm ở chế độ `eval()` trong suốt quá trình train để giữ cố định các thống kê chuẩn hóa của ImageNet.
- **Head Warmup:** Trong các epoch đầu (`c.warmup = 2`), toàn bộ backbone được đóng băng, chỉ cập nhật lớp Head với tốc độ học cao hơn ($3 \times 10^{-4}$) ; với mode full, sau warmup mở khóa backbone với tốc độ học nhỏ ($2 \times 10^{-5}$).

Giữ thống kê BatchNorm không có nghĩa đóng băng mọi tham số của lớp đó: hệ số scale/shift vẫn học khi backbone được mở khóa. Đây là quy ước được giữ giống nhau khi so sánh, chưa phải kết luận rằng nó luôn tốt hơn cập nhật thống kê.

Tensor là mảng nhiều chiều của PyTorch. Batch này có các trục [số cặp, 2 ảnh, số crop, 3 kênh, cao, rộng]. `backward()` tính gradient của loss theo tham số; optimizer dùng gradient để cập nhật. Microbatch là số cặp xử lý một lượt; effective batch có thể tích lũy nhiều lượt trước một lần cập nhật. Demo chỉ cập nhật một batch 2 cặp, không thay thế thí nghiệm đầy đủ.

## 3. Trực quan hóa bước `train_step` trên dữ liệu thật

Chúng ta lấy một microbatch gồm 2 cặp ảnh thật từ tập TRAIN Fold 0 để quan sát tường minh:

In [ ]:
data_root_env = os.environ.get('DATA_ROOT')
data_root = Path(data_root_env or TASK_ROOT / 'data/train').expanduser().resolve()
if not (data_root / 'pairs.csv').is_file():
    raise FileNotFoundError(f'Thiếu dữ liệu train: {data_root / "pairs.csv"}. Xem README để đặt DATA_ROOT.')

if (data_root / 'pairs.csv').is_file():
    dev_frame = prepare_development(data_root)
    tr_fold0, va_fold0 = split_fold(dev_frame, fold=0)
    
    # Nạp cấu hình center60
    c_demo = Config(**read_json(TASK_ROOT / 'configs/center60.json'))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Tạo mô hình demo từ trọng số ImageNet
    demo_model, _ = model_for(c_demo, device=device, pretrained=True)
    demo_model.train()
    train_mode(demo_model, c_demo, epoch=1)
    
    # Lấy 1 batch gồm 2 cặp ảnh thật
    demo_ds = Pairs(tr_fold0.head(2), data_root, c_demo, training=True, epoch=1)
    demo_x, demo_y = next(iter(make_loader(demo_ds, batch=2)))
    
    demo_x = demo_x.to(device)
    demo_y = demo_y.to(device)
    
    # 1. Forward và tính Logits
    with torch.no_grad():
        img_logits = logits(demo_model, demo_x, c_demo)
        s0, s1 = img_logits[:, 0], img_logits[:, 1]
        pair_prob = torch.sigmoid(s1 - s0)
        target_labels = torch.stack([1 - demo_y, demo_y], dim=1)
        loss_val = objective(img_logits, demo_y, c_demo)
        
    print("=== CÁC GIÁ TRỊ TÍNH TOÁN TRÊN BATCH THẬT ===")
    print(f"Vector nhãn cặp [1-y, y]:\n{target_labels.cpu().numpy()}")
    print(f"Logits từng ảnh [s0, s1]:\n{img_logits.cpu().numpy()}")
    print(f"Xác suất cặp sigmoid(s1 - s0): {pair_prob.cpu().numpy()}")
    print(f"Giá trị Loss (BCE): {loss_val.item():.4f}")
    
    # 2. Hiển thị mã nguồn hàm train_step dùng chung
    print("\n=== MÃ NGUỒN HÀM train_step CHIA SẺ ===")
    print(inspect.getsource(train_step))
    
    # 3. Thực hiện bước cập nhật
    print('Tensor x/y:', tuple(demo_x.shape), tuple(demo_y.shape))
    print(inspect.getsource(model_for))
    print(inspect.getsource(objective))
    head = head_name(c_demo)
    demo_opt = torch.optim.AdamW([
        {'params': [p for name, p in demo_model.named_parameters() if not name.startswith(head)], 'lr': c_demo.backbone_lr},
        {'params': [p for name, p in demo_model.named_parameters() if name.startswith(head)], 'lr': c_demo.head_lr},
    ], weight_decay=c_demo.weight_decay)
    demo_opt.zero_grad(set_to_none=True)
    step_loss, preclip_norm = train_step(
        model=demo_model, x=demo_x, y=demo_y, optimizer=demo_opt,
        c=c_demo, target_batch=2, clip_grad=5.0, perform_update=True, device=device
    )
    print(f"Thực thi train_step: Loss = {step_loss:.4f}, Gradient norm (trước khi clip) = {preclip_norm:.4f}")
    
    # Hủy mô hình demo trước khi huấn luyện chính thức
    del demo_model, demo_opt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 4. Huấn luyện đối chứng trên Fold 0: Frozen vs center60

Ta huấn luyện hai mô hình:
- **`frozen`:** Đóng băng backbone trong toàn bộ quá trình train (tối đa 24 epoch, có early stopping), chỉ huấn luyện lớp Head.
- **`center60`:** Warmup 2 epoch head, sau đó tinh chỉnh toàn bộ mạng.

Hai nhánh cùng split, seed, crop center60 và giới hạn epoch. Yếu tố thay đổi là tham số backbone có được học sau warmup hay không. Epoch là một lượt qua tập train; early stopping dừng khi validation không cải thiện đủ lâu.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Bước huấn luyện CNN cần CUDA. Bạn có thể chạy nhánh LR ở bài 01 và 05 trên CPU.")

if (data_root / 'pairs.csv').is_file() and torch.cuda.is_available():
    session_dir = load_session(RUN_ID, data_root)
    print(f"Nạp session tại: {session_dir.name}")
    
    # 1. Huấn luyện / Nạp Frozen Fold 0
    folder_frozen, res_frozen = train_cnn_fold('frozen', dev_frame, data_root, session_dir, fold=0)
    
    # 2. Huấn luyện / Nạp center60 Fold 0
    folder_full, res_full = train_cnn_fold('center60', dev_frame, data_root, session_dir, fold=0)
    
    print("\n=== KẾT QUẢ SO SÁNH VALIDATION FOLD 0 ===")
    df_compare = pd.DataFrame([
        {'Cấu hình': '1. Frozen Backbone (Head only)', **res_frozen['metrics']},
        {'Cấu hình': '2. Full Fine-tuning (center60)', **res_full['metrics']},
    ])
    print(df_compare[['Cấu hình', 'macro_f1', 'accuracy', 'log_loss', 'errors']].to_string(index=False))
else:
    print("Huấn luyện CNN yêu cầu môi trường GPU CUDA và dataset tại data/train.")

## 5. Phân tích Lỗi trên tập Validation Fold 0

So sánh chi tiết các mẫu dự đoán giữa hai mô hình:

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Bước huấn luyện CNN cần CUDA. Bạn có thể chạy nhánh LR ở bài 01 và 05 trên CPU.")

if (data_root / 'pairs.csv').is_file() and torch.cuda.is_available():
    pred_f = read_csv(folder_frozen / 'development.csv').rename(columns={'y': 'fake_position', 'fold': 'inner_fold'})
    pred_c = read_csv(folder_full / 'development.csv').rename(columns={'y': 'fake_position', 'fold': 'inner_fold'})
    
    pred_f = aligned_predictions(pred_f, va_fold0)
    pred_c = aligned_predictions(pred_c, va_fold0)
    
    y_true = va_fold0.fake_position.to_numpy()
    corr_f = (pred_f.p.to_numpy() >= 0.5) == y_true
    corr_c = (pred_c.p.to_numpy() >= 0.5) == y_true
    
    fixed_by_full = (~corr_f) & corr_c
    new_err_by_full = corr_f & (~corr_c)
    
    print(f"Số lỗi mô hình Frozen mắc phải:       {np.sum(~corr_f)}")
    print(f"Số lỗi mô hình center60 mắc phải:      {np.sum(~corr_c)}")
    print(f"Số lỗi được sửa khi fine-tune full:    {np.sum(fixed_by_full)}")
    print(f"Số lỗi mới xuất hiện ở fine-tune full: {np.sum(new_err_by_full)}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show_case(row, caption):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for side, ax in enumerate(axes):
        with Image.open(data_root / row[f'image_{side}']) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(f'image_{side}')
        ax.axis('off')
    fig.suptitle(f"{caption} | ID={row.pair_id} | ảnh giả={row.fake_position}")
    plt.tight_layout()
    plt.show()

for title, mask in [('Full sửa lỗi Frozen', fixed_by_full), ('Full tạo lỗi mới so với Frozen', new_err_by_full)]:
    cases = va_fold0.loc[mask].sort_values('pair_id')
    if cases.empty:
        print(title, ': không có mẫu')
        continue
    row = cases.iloc[0]
    i = va_fold0.index[va_fold0.pair_id == row.pair_id][0]
    show_case(row, f'{title}: p_frozen={pred_f.p.iloc[i]:.3f}, p_full={pred_c.p.iloc[i]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, folder in [('frozen', folder_frozen), ('center60', folder_full)]:
    history = pd.read_csv(folder / 'history.csv')
    axes[0].plot(history.epoch, history.train_loss, label=label)
    axes[1].plot(history.epoch, history.dev_f1, label=label)
axes[0].set(xlabel='Epoch', ylabel='Train loss')
axes[1].set(xlabel='Epoch', ylabel='Validation Macro-F1')
for ax in axes: ax.legend()
plt.show()

## Từ kết quả đến câu hỏi tiếp theo

Full fine-tuning có tăng Macro-F1 và giảm tổng số lỗi trên lần chạy này không? Đọc cả lỗi được sửa lẫn lỗi mới, rồi đối chiếu đường train loss/validation: loss train giảm chưa bảo đảm validation tăng. Không khẳng định fine-tuning tốt hơn trước khi có kết quả.

Cả hai nhánh cùng crop và resize. Bài 03 giữ crop cố định để hỏi riêng việc giảm độ phân giải rồi phóng lại làm thay đổi dự đoán thế nào.